# MODI-SYNTH · Baseline Reader — **SPAN edition**

This notebook stops improvising and implements **SPAN** (Coquenet et al., 2021), the published
method for our exact problem: read a paragraph when you only have paragraph-level transcription
and no line labels. SPAN reports **6.46% CER on IAM trained from scratch on paragraphs**.

**Run it with:** GPU T4 x2 · Internet On · *Save Version → Save & Run All (Commit)*.

### What changed from notebook 03, and why

| | notebook 03 | here | source |
|---|---|---|---|
| epochs | 250 | **5,000** (across sessions) | SPAN `max_nb_epochs: 5000` |
| recurrence | 2-layer BiLSTM over ~1,000 steps | **none — fully convolutional** | SPAN is recurrence-free |
| input | lines cut out and laid end to end | **the whole patch, untouched** | SPAN needs no segmentation |
| learning rate | 3e-4 AdamW + cosine | **1e-4 Adam, constant** | SPAN `lr: 0.0001` |
| batch | 4 | **8** | SPAN `batch_size: 8` |
| dropout | 0.1 | **0.5** | SPAN `dropout: 0.5` |
| channels | 256 | **512** | SPAN FCN_Encoder |
| augmentation | 0.3–0.5 per transform | **0.2 per transform** | SPAN augmentation config |

### Why the earlier runs looked like they plateaued

They were not plateaus. SPAN and VAN both train for **5,000 epochs**; we had never passed 250, and
at 179 s/epoch an 11 h session only buys ~220. The BiLSTM was the reason an epoch cost that much —
it runs sequentially over ~1,000 timesteps. Removing it is what makes the required epoch count
affordable, which is precisely why SPAN is recurrence-free.

### Running past 12 hours
Kaggle stops a commit at 12 h. `last.pt` is written every epoch and reloaded on start, so to keep
going: *Add Input* → add **this notebook's own previous output** → commit again. Cell 1 restores
`runs/` and training continues at the next epoch with the optimiser state intact.

> SPAN: Coquenet, Chatelain & Paquet, *SPAN: a Simple Predict & Align Network for Handwritten
> Paragraph Recognition*, ICDAR 2021. arXiv:2102.08742. Code CC-BY-4.0.

In [ ]:
# Cell 1 - environment, GPU, wall-clock budget, and pick up any earlier session's progress
import os, sys, time, json, re, math, random, csv, glob, shutil, zipfile, pathlib
T_START = time.time()
BUDGET_H = 11.0          # Kaggle kills a commit at 12 h; stop at 11 h so the last cells still run
def elapsed_h(): return (time.time() - T_START) / 3600
def budget_left(): return BUDGET_H - elapsed_h()

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on the GPU: Settings -> Accelerator -> GPU T4 x2'
print('GPU:', torch.cuda.get_device_name(0))
print(f'budget: {BUDGET_H} h, then it stops and saves whatever it has')

# RESUME: if a previous commit's output is attached as an input dataset, copy its runs/ folder
# here. fit() then finds last.pt and carries straight on. This is how we reach 5,000 epochs.
restored = 0
for cand in glob.glob('/kaggle/input/*/runs/*/last.pt'):
    dst = '/kaggle/working/runs/' + cand.split('/runs/', 1)[1]
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for f in glob.glob(os.path.join(os.path.dirname(cand), '*')):
        shutil.copyfile(f, os.path.join(os.path.dirname(dst), os.path.basename(f)))
    ck = torch.load(dst, map_location='cpu', weights_only=False)
    print(f"RESUMING {cand} -> epoch {ck['state']['epoch']}, best val CER {ck['state']['best_cer']:.2f}")
    restored += 1
if not restored:
    print('fresh start (no previous run attached as an input)')

In [ ]:
# Cell 2 - configuration. Every value is SPAN's unless a comment says otherwise.
#   SPAN training script : lr 1e-4 Adam, batch 8, 5000 epochs, dropout 0.5, no mixed precision
#   SPAN FCN_Encoder     : 1->32->64->128->256->512->512, InstanceNorm, height/32, width/8
# Our ONE deviation is the input height (see IMG_H below), and it is forced by our data.
OUT = '/kaggle/working'

# SPAN resizes pages to 150 DPI. Our patches are crops of 3-6 text lines, not full pages, so a
# fixed DPI is meaningless here; we fix the height instead. 320 is chosen so the output grid lands
# in SPAN's own working regime of roughly 10-15 output slots per character:
#   height 320 -> 320/32 = 10 output rows for 3-6 text lines (SPAN uses ~2.5 rows per line)
#   a typical patch is 2.74x wider than tall -> ~877 px wide -> 877/8 = ~110 columns
#   10 x 110 = ~1,100 slots for a median 91-character label = ~12 slots per character
IMG_H = 320

CFG = dict(
    run_name = 'SPAN_seed42', seed = 42,

    img_h = IMG_H, max_w = 2048,      # max_w is a safety cap only; width is padded per batch
    batch_size = 8,                   # SPAN: batch_size 8
    lr = 1e-4,                        # SPAN: lr 0.0001, Adam, amsgrad False
    max_epochs = 5000,                # SPAN: max_nb_epochs 5000
    patience = 400,                   # SPAN stops on time/epochs; this only catches a dead run
    min_delta = 0.05,
    dropout = 0.5,                    # SPAN: dropout 0.5
    d_model = 512,                    # SPAN FCN_Encoder final channels
    aug_prob = 0.2,                   # SPAN: each augmentation at probability 0.2
    use_aug = True,
    num_workers = 2,
)
RUN_DIR = f"{OUT}/runs/{CFG['run_name']}"
os.makedirs(RUN_DIR, exist_ok=True)
print(f"input height {CFG['img_h']} px, width padded per batch (cap {CFG['max_w']})")
print(f"target: {CFG['max_epochs']} epochs across however many sessions that takes")
print(f"run dir: {RUN_DIR}")

In [ ]:
# Cell 3 - libraries Kaggle does not ship
!pip -q install datasets editdistance sacrebleu 2>&1 | tail -1
print('installed')

In [ ]:
# Cell 4 - find colab_pack.zip anywhere under /kaggle/input and unpack the metadata
DATA = '/kaggle/working/data'
os.makedirs(DATA, exist_ok=True)
need = ['splits.json', 'modetrans_text.csv', 'charset.json']

def find_metadata():
    for root, _, files in os.walk('/kaggle/input'):
        if all(n in files for n in need):
            return root
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip'):
                zipfile.ZipFile(os.path.join(root, f)).extractall(DATA)
                for r2, _, f2 in os.walk(DATA):
                    if all(n in f2 for n in need):
                        return r2
    return None

META = find_metadata()
assert META, ('Could not find the metadata. Add colab_pack.zip as an input dataset '
              '(right panel -> Add Input -> Upload).')
print('metadata:', META, sorted(os.listdir(META))[:6])

In [ ]:
# Cell 5 - MoDeTrans images + our fixed splits and cleaned text (unchanged from notebook 03,
# so the test numbers stay comparable with everything measured so far)
import pandas as pd
from datasets import load_dataset

ds = load_dataset('historyHulk/MoDeTrans', split='train')     # 2,043 rows
names  = list(ds['filename'])
splits = json.load(open(f'{META}/splits.json', encoding='utf-8'))['modetrans']['assignment']
text   = (pd.read_csv(f'{META}/modetrans_text.csv', keep_default_na=False)
            .set_index('image_id')['text_train'].to_dict())

assert len(names) == 2043 and set(names) == set(splits) == set(text), 'dataset / split mismatch'
SPLIT_IDS = {s: [n for n in names if splits[n] == s] for s in ['train', 'val', 'test']}
assert not (set(SPLIT_IDS['train']) & (set(SPLIT_IDS['val']) | set(SPLIT_IDS['test']))), 'leakage!'
print('splits:', {k: len(v) for k, v in SPLIT_IDS.items()})

In [ ]:
# Cell 6 - alphabet: index 0 is the CTC blank, then the 86 symbols, then <FRAC> and <unk>
cs = json.load(open(f'{META}/charset.json', encoding='utf-8'))
VOCAB = ['<blank>'] + cs['symbols'] + ['<FRAC>', '<unk>']
STOI = {c: i for i, c in enumerate(VOCAB)}
NUM_CLASSES = len(VOCAB)
_TOK = re.compile(r'<FRAC>|.', re.S)
encode     = lambda s: [STOI.get(t, STOI['<unk>']) for t in _TOK.findall(s)]
decode_ids = lambda ids: ''.join(VOCAB[i] for i in ids if i > 0)

bad = [n for n in names if decode_ids(encode(text[n])) != text[n]]
assert not bad, f'round-trip failed: {bad[:5]}'
assert sum(t == STOI['<unk>'] for n in SPLIT_IDS['train'] for t in encode(text[n])) == 0
MAX_LEN = max(len(encode(text[n])) for n in names)
print(f'{NUM_CLASSES} classes, longest label {MAX_LEN} characters')

In [ ]:
# Cell 7 - images. THE WHOLE PATCH, scaled to a fixed height. Nothing is cut, split,
# deskewed, reordered or thrown away.
#
# This is the single biggest change from notebook 03, and it is what SPAN prescribes. Notebook 03
# tried to find the text lines with a projection profile, crop each one, and lay them end to end.
# That created a long chain of failure points we spent days chasing: pages whose lines merged into
# one band, lines split in half, header lines dropped, ink thrown away, and a reading order that
# depended entirely on the segmenter being right. SPAN does none of it. It feeds the paragraph in
# as it is and lets the network learn the alignment - that is what "Predict & Align" means, and it
# is why the paper needs no line labels. Every bug in that pipeline disappears with it.
import numpy as np, cv2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

H, MAXW = CFG['img_h'], CFG['max_w']

def prepare(pil):
    # greyscale, scale to height H keeping the aspect ratio, return the image and its width
    g = np.array(pil.convert('L'))
    w = max(8, int(round(g.shape[1] * H / g.shape[0])))
    if w > MAXW:
        w = MAXW
    return cv2.resize(g, (w, H), interpolation=cv2.INTER_AREA), w

CACHE, CACHEW = f'{OUT}/pages_{H}.npy', f'{OUT}/pages_{H}_w.npy'
if os.path.exists(CACHE) and os.path.exists(CACHEW):
    IMGS, WID = np.load(CACHE), np.load(CACHEW)
else:
    res = [prepare(r['image']) for r in tqdm(ds, desc=f'scaling every patch to height {H}')]
    WID = np.array([w for _, w in res], np.int32)
    IMGS = np.full((len(res), H, int(WID.max())), 255, np.uint8)
    for i, (im, w) in enumerate(res):
        IMGS[i, :, :w] = im
    np.save(CACHE, IMGS); np.save(CACHEW, WID)
IDX = {n: i for i, n in enumerate(names)}

# The output grid is (H/32) rows by (W/8) columns, so the number of CTC slots is exactly
# (H/32)*(W/8). CTC needs at least one slot per label character, plus one more wherever the label
# repeats a character. Check every patch, because a sample that fails this returns ZERO loss
# silently and teaches the model nothing.
ROWS = H // 32
SLOTS = (ROWS * (WID // 8)).astype(int)
NEED = np.array([len(encode(text[n])) + sum(a == b for a, b in zip(encode(text[n]), encode(text[n])[1:]))
                 for n in names])
LAB = np.array([len(encode(text[n])) for n in names])
tight = SLOTS < NEED
print(f'widths: median {np.median(WID):,.0f} px, max {WID.max():,} (cap {MAXW:,})')
print(f'grid: {ROWS} rows x (width/8) columns')
print(f'slots per patch: median {np.median(SLOTS):,.0f} | slots per character: '
      f'median {np.median(SLOTS/np.maximum(1,LAB)):.1f}  (SPAN works at roughly 10-15)')
if tight.any():
    print(f'WARNING: {tight.sum()} patches have fewer slots than their label needs; '
          f'widening them so CTC can align')
    for i in np.where(tight)[0]:
        want = min(MAXW, int(NEED[i] * 8 / ROWS * 1.3) + 8)
        IMGS[i, :, :want] = cv2.resize(IMGS[i][:, :WID[i]], (want, H))
        WID[i] = want
    np.save(CACHE, IMGS); np.save(CACHEW, WID)
    SLOTS = (ROWS * (WID // 8)).astype(int)
assert (SLOTS >= NEED).all(), 'some patches still cannot fit their label'
print('every patch has room for its label')
print('cached', IMGS.shape, f'{IMGS.nbytes/1e9:.2f} GB')

ex = IDX[SPLIT_IDS['train'][0]]
plt.figure(figsize=(14, 4)); plt.imshow(IMGS[ex][:, :WID[ex]], cmap='gray'); plt.axis('off')
plt.title(f'what the model sees: the whole patch at {H} px tall, {WID[ex]} px wide'); plt.show()

In [ ]:
# Cell 8 - augmentation and data loading.
# SPAN applies each transform independently at probability 0.2: DPI change, perspective, elastic
# distortion, random transform, dilation/erosion, brightness and contrast. These are the same
# families notebook 03 used, dialled back from 0.3-0.5 to SPAN's 0.2.
from torch.utils.data import Dataset, DataLoader, Sampler

P = CFG['aug_prob']

def augment(img, rng):
    h, w = img.shape
    out = img
    if rng.random() < P:                                    # DPI change (SPAN: factor 0.75-1.0)
        f = rng.uniform(0.75, 1.0)
        small = cv2.resize(out, (max(8, int(w*f)), max(8, int(h*f))), interpolation=cv2.INTER_AREA)
        out = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    if rng.random() < P:                                    # perspective
        m = 0.3 * min(h, w) * 0.1
        src = np.float32([[0,0],[w,0],[w,h],[0,h]])
        dst = src + rng.uniform(-m, m, src.shape).astype(np.float32)
        out = cv2.warpPerspective(out, cv2.getPerspectiveTransform(src, dst), (w, h),
                                  borderValue=255)
    if rng.random() < P:                                    # elastic distortion
        a, s = rng.uniform(10, 20), rng.uniform(4, 6)
        dx = cv2.GaussianBlur((rng.random((h, w))*2-1).astype(np.float32), (0,0), s)*a
        dy = cv2.GaussianBlur((rng.random((h, w))*2-1).astype(np.float32), (0,0), s)*a
        gx, gy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
        out = cv2.remap(out, gx+dx, gy+dy, cv2.INTER_LINEAR, borderValue=255)
    if rng.random() < P:                                    # small affine
        M = cv2.getRotationMatrix2D((w/2, h/2), rng.uniform(-1.0, 1.0), rng.uniform(0.98, 1.02))
        out = cv2.warpAffine(out, M, (w, h), flags=cv2.INTER_LINEAR, borderValue=255)
    if rng.random() < P:                                    # dilation / erosion (SPAN: kernel 1-3)
        k = np.ones((rng.integers(1, 4), rng.integers(1, 4)), np.uint8)
        out = cv2.erode(out, k) if rng.random() < 0.5 else cv2.dilate(out, k)
    if rng.random() < P:                                    # brightness / contrast
        out = np.clip(out.astype(np.float32)*rng.uniform(0.9, 1.1) + rng.uniform(-20, 20),
                      0, 255).astype(np.uint8)
    return out

W_MULT = 8          # the encoder divides width by 8; pad to a multiple so the slot count is exact

class DS(Dataset):
    def __init__(s, ids, train, aug): s.ids, s.train, s.aug = ids, train, aug
    def __len__(s): return len(s.ids)
    def __getitem__(s, i):
        n = s.ids[i]; j = IDX[n]
        img = IMGS[j][:, :WID[j]]
        if s.train and s.aug:
            img = augment(img, np.random.default_rng(torch.randint(0, 2**31-1, (1,)).item()))
        x = torch.from_numpy(np.ascontiguousarray(img)).float().div(255).sub(0.5).div(0.5).unsqueeze(0)
        return x, torch.tensor(encode(text[n]), dtype=torch.long), n

def collate(b):
    xs, ys, ns = zip(*b)
    ws = [x.shape[-1] for x in xs]
    Wmax = int(math.ceil(max(ws) / W_MULT) * W_MULT)
    out = torch.full((len(xs), 1, CFG['img_h'], Wmax), 1.0)   # 1.0 == white after normalising
    for i, x in enumerate(xs): out[i, :, :, :x.shape[-1]] = x
    return (out, torch.cat(ys), torch.tensor([len(y) for y in ys]),
            torch.tensor(ws, dtype=torch.long), list(ns))

class WidthBucketSampler(Sampler):
    # Group patches of similar width into the same batch so almost nothing is padding, while the
    # ORDER of batches is still shuffled every epoch.
    def __init__(s, ids, bs, shuffle, drop_last):
        s.ids, s.bs, s.shuffle, s.drop_last = list(ids), bs, shuffle, drop_last
    def __iter__(s):
        order = sorted(range(len(s.ids)), key=lambda i: (WID[IDX[s.ids[i]]],
                       torch.randint(0, 10**6, (1,)).item() if s.shuffle else 0))
        batches = [order[i:i+s.bs] for i in range(0, len(order), s.bs)]
        if s.drop_last and len(batches) > 1 and len(batches[-1]) < s.bs: batches.pop()
        if s.shuffle:
            batches = [batches[i] for i in torch.randperm(len(batches)).tolist()]
        return iter(batches)
    def __len__(s):
        n = (len(s.ids) + s.bs - 1) // s.bs
        return n - 1 if (s.drop_last and len(s.ids) % s.bs) else n

def make_loader(ids, train, cfg=None):
    cfg = cfg or CFG; bs, nw = cfg['batch_size'], cfg['num_workers']
    return DataLoader(DS(ids, train, cfg['use_aug']),
                      batch_sampler=WidthBucketSampler(ids, bs, train,
                                                       bool(train) and len(ids) > bs),
                      num_workers=nw, collate_fn=collate, pin_memory=True,
                      persistent_workers=nw > 0)

# How much of each batch is padding? The loss now covers the whole padded grid, so this needs to
# stay small. Width bucketing is what keeps it there.
_w = np.array([WID[IDX[n]] for n in SPLIT_IDS['train']], float)
_sam = WidthBucketSampler(SPLIT_IDS['train'], CFG['batch_size'], True, True)
_bat = [[_w[i] for i in b] for b in _sam]
_pad = float(np.mean([1 - np.mean(b)/max(b) for b in _bat]))
print(f'padding added by width-bucketed batching: {100*_pad:.1f}% of each batch '
      f'(mean padded width {np.mean([max(b) for b in _bat]):,.0f} px)')
assert _pad < 0.10, f'{100*_pad:.0f}% padding is too much for a full-grid CTC length'

_j = IDX[SPLIT_IDS['train'][1]]
fig, ax = plt.subplots(1, 2, figsize=(15, 3))
ax[0].imshow(IMGS[_j][:, :WID[_j]], cmap='gray'); ax[0].set_title('original'); ax[0].axis('off')
ax[1].imshow(augment(IMGS[_j][:, :WID[_j]], np.random.default_rng(0)), cmap='gray')
ax[1].set_title('augmented'); ax[1].axis('off'); plt.show()
print('data pipeline ready')

In [ ]:
# Cell 9 - VISUAL AUDIT. Far shorter than notebook 03's, because there is far less to check:
# the only thing done to an image now is a scale to height 320. Nothing is cut or reordered, so
# there is no segmentation to get wrong. Saved to /kaggle/working/image-processing-audit/.
AUD = f'{OUT}/image-processing-audit'
os.makedirs(AUD, exist_ok=True)

spc = SLOTS / np.maximum(1, LAB)
CHECKS = {
    'hit_the_width_cap'       : (WID >= MAXW, f'scaled down to fit the {MAXW} px cap'),
    'too_few_slots_for_label' : (SLOTS < NEED, 'CTC cannot align; returns ZERO loss silently'),
    'slots_per_char_below_3'  : (spc < 3.0,  'too few output slots per character'),
    'slots_per_char_above_40' : (spc > 40.0, 'very sparse; harmless, just more blanks to learn'),
}
rows = []
print(f'{"check":28s} {"flagged":>8s}   meaning')
for k, (m, why) in CHECKS.items():
    print(f'{k:28s} {int(m.sum()):8d}   {why}')
    rows.append((k, int(m.sum()), round(100*float(m.mean()), 2), why))
pd.DataFrame(rows, columns=['check','n_flagged','pct','meaning']).to_csv(f'{AUD}/checks.csv', index=False)
pd.DataFrame({'image_id': names, 'width': WID, 'slots': SLOTS, 'label_len': LAB,
              'slots_per_char': spc.round(2)}).to_csv(f'{AUD}/stats.csv', index=False)

_rs = np.random.default_rng(42)
for n in [names[i] for i in _rs.choice(len(names), 6, replace=False)]:
    j = IDX[n]
    f, a = plt.subplots(2, 1, figsize=(14, 6))
    a[0].imshow(IMGS[j][:, :WID[j]], cmap='gray'); a[0].axis('off')
    a[0].set_title(f'{n}: the whole patch at {H}x{WID[j]} - this is the model input, unmodified',
                   fontsize=8, loc='left')
    a[1].imshow(augment(IMGS[j][:, :WID[j]], np.random.default_rng(7)), cmap='gray'); a[1].axis('off')
    a[1].set_title(f'the same patch with SPAN augmentation | {SLOTS[j]} output slots for a '
                   f'{LAB[j]}-character label = {spc[j]:.1f} per character', fontsize=8, loc='left')
    f.tight_layout(); f.savefig(f'{AUD}/{n.replace(".jpg","")}__input_and_augmented.png',
                                dpi=100, bbox_inches='tight'); plt.close(f)
print(f'\nwrote {len([x for x in os.listdir(AUD) if x.endswith(".png")])} images + checks.csv '
      f'+ stats.csv to {AUD}')

In [ ]:
# Cell 10 - the model: SPAN's FCN_Encoder, reproduced from the paper and the authors' models.py.
#
#   ConvBlocks  1->32 (1,1), 32->64 (2,2), 64->128 (2,2), 128->256 (2,2), 256->512 (2,1), 512->512 (2,1)
#               height: 1*2*2*2*2*2 = 32      width: 1*2*2*2*1*1 = 8
#   then 4 depthwise-separable residual blocks at 512 channels
#   InstanceNorm2d(eps=1e-3) throughout, MixDropout (Dropout 0.5 + Dropout2d 0.25)
#   a 1x1 convolution predicts character probabilities at every position of the 2D grid
#   the 2D grid is flattened to a 1D CTC sequence by concatenating ROWS, top to bottom
#
# There is NO recurrence. That is deliberate in the paper and it is the reason this is affordable:
# the BiLSTM in notebook 03 ran sequentially over ~1,000 timesteps and made an epoch cost 179 s,
# which caps an 11 h session at ~220 epochs when the method needs 5,000.
import torch.nn as nn, torch.nn.functional as F

class MixDropout(nn.Module):
    def __init__(s, p=0.5, p2d=0.25):
        super().__init__(); s.d, s.d2 = nn.Dropout(p), nn.Dropout2d(p2d)
    def forward(s, x): return s.d(x) if random.random() < 0.5 else s.d2(x)

def norm(c): return nn.InstanceNorm2d(c, eps=1e-3, affine=True)

class ConvBlock(nn.Module):
    def __init__(s, cin, cout, stride, drop):
        super().__init__()
        s.c1 = nn.Conv2d(cin, cout, 3, padding=1)
        s.c2 = nn.Conv2d(cout, cout, 3, padding=1)
        s.c3 = nn.Conv2d(cout, cout, 3, stride=stride, padding=1)
        s.n, s.dp = norm(cout), MixDropout(drop, drop/2)
    def forward(s, x):
        x = F.relu(s.c1(x)); x = s.dp(x)
        x = F.relu(s.c2(x)); x = s.n(x); x = s.dp(x)
        return F.relu(s.c3(x))

class DSCBlock(nn.Module):
    def __init__(s, c, drop):
        super().__init__()
        s.c1 = nn.Sequential(nn.Conv2d(c, c, 3, padding=1, groups=c), nn.Conv2d(c, c, 1))
        s.c2 = nn.Sequential(nn.Conv2d(c, c, 3, padding=1, groups=c), nn.Conv2d(c, c, 1))
        s.n, s.dp = norm(c), MixDropout(drop, drop/2)
    def forward(s, x):
        y = F.relu(s.c1(x)); y = s.dp(y)
        y = F.relu(s.c2(y)); y = s.n(y); y = s.dp(y)
        return x + y

class SPAN(nn.Module):
    def __init__(s, n_classes, d=512, drop=0.5):
        super().__init__()
        s.enc = nn.Sequential(
            ConvBlock(1, 32, (1,1), drop), ConvBlock(32, 64, (2,2), drop),
            ConvBlock(64, 128, (2,2), drop), ConvBlock(128, 256, (2,2), drop),
            ConvBlock(256, d, (2,1), drop), ConvBlock(d, d, (2,1), drop),
            DSCBlock(d, drop), DSCBlock(d, drop), DSCBlock(d, drop), DSCBlock(d, drop))
        s.head = nn.Conv2d(d, n_classes, 1)
    def forward(s, x):
        f = s.enc(x)
        lo = s.head(f)                                  # B, C, rows, cols
        B, C, Hh, Ww = lo.shape
        # flatten the 2D grid to one CTC sequence by concatenating rows top to bottom
        return lo.permute(0, 2, 3, 1).reshape(B, Hh * Ww, C), (Hh, Ww)

DEV = 'cuda'
_m = SPAN(NUM_CLASSES, CFG['d_model'], CFG['dropout']).to(DEV).eval()
with torch.no_grad():
    _s, _g = _m(torch.zeros(1, 1, CFG['img_h'], 880, device=DEV))
print(f'params {sum(p.numel() for p in _m.parameters())/1e6:.1f} M  (SPAN is ~19 M)')
print(f'an 880 px wide patch -> grid {_g[0]} rows x {_g[1]} cols = {_s.shape[1]} CTC slots')
assert _g[0] == CFG['img_h'] // 32, 'height must divide by 32'
assert _g[1] == 880 // 8, 'width must divide by 8'
del _m, _s; torch.cuda.empty_cache()
print('model OK')

In [ ]:
# Cell 11 - decoding and metrics. Unchanged from notebook 03 so every number stays comparable.
import editdistance, sacrebleu
_FR = ''
_u = lambda s: s.replace('<FRAC>', _FR)

def greedy_decode(seq):
    ids = seq.argmax(-1).cpu().numpy()
    keep = np.ones_like(ids, dtype=bool); keep[:,1:] = ids[:,1:] != ids[:,:-1]; keep &= ids != 0
    return [decode_ids(r[k]) for r, k in zip(ids, keep)]

def blank_share(seq):
    ids = seq.argmax(-1).cpu().numpy()
    return float((ids == 0).mean())

def compute_metrics(refs, hyps, n_boot=0, seed=0):
    e = np.array([editdistance.eval(_u(r), _u(h)) for r, h in zip(refs, hyps)], float)
    L = np.array([max(1, len(_u(r))) for r in refs], float)
    we = np.array([editdistance.eval(r.split(), h.split()) for r, h in zip(refs, hyps)], float)
    wl = np.array([max(1, len(r.split())) for r in refs], float)
    per = e / L
    out = dict(CER=100*e.sum()/L.sum(), WER=100*we.sum()/wl.sum(),
               median_CER=100*float(np.median(per)), catastrophic_pct=100*float((per>0.5).mean()),
               chrF_pp=sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score,
               BLEU=sacrebleu.corpus_bleu(hyps, [refs]).score, n=len(refs))
    if n_boot:
        rg = np.random.default_rng(seed); idx = rg.integers(0, len(refs), (n_boot, len(refs)))
        b = 100*e[idx].sum(1)/L[idx].sum(1)
        out['CER_CI95'] = (float(np.percentile(b,2.5)), float(np.percentile(b,97.5)))
    return out
print('metrics self-test:', {k: round(v,2) for k,v in compute_metrics(['राम आला'],['राम आल']).items() if k!='n'})

In [ ]:
# Cell 12 - training loop. SPAN's optimiser settings, plus the stability guards that earlier
# runs earned the hard way:
#   * full fp32 (SPAN sets use_amp False too). Mixed precision overflowed and silently killed a run.
#   * a batch with a non-finite loss is skipped; if more than 5% of an epoch is skipped it ABORTS
#   * gradient and weight norms are logged so instability is visible before it is fatal
#   * last.pt is written EVERY epoch, so a session that runs out of time resumes exactly
#   * NO cold-start abort. It killed two healthy runs. SPAN needs thousands of epochs and the
#     early ones legitimately sit near the letter-frequency prior (3.32 nats).
CTC = nn.CTCLoss(blank=0, zero_infinity=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# CTC input lengths: the FULL grid, the same for every sample in the batch.
# This is not laziness, it is forced by the geometry. The 2D grid is flattened row-major, so a
# sample narrower than the batch maximum occupies the first columns of EVERY row - its valid cells
# are scattered through the sequence, not sitting in a prefix. A per-sample length would therefore
# slice the wrong cells. SPAN pads the batch and uses the whole grid, and the width-bucketed
# sampler keeps that padding to a fraction of a percent (printed in cell 8).
def full_lens(seq):
    return torch.full((seq.size(0),), seq.size(1), dtype=torch.long)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); refs, hyps, ns, bl, nb = [], [], [], 0.0, 0
    for x, y, yl, ws, n in loader:
        seq, g = model(x.to(DEV, non_blocking=True))
        hyps += greedy_decode(seq); refs += [text[k] for k in n]; ns += n
        bl += blank_share(seq); nb += 1
    return refs, hyps, ns, bl/max(1, nb)

def fit(cfg, train_ids, val_ids, run_dir, verbose_every=1, stop_at_cer=None):
    os.makedirs(run_dir, exist_ok=True); set_seed(cfg['seed'])
    model = SPAN(NUM_CLASSES, cfg['d_model'], cfg['dropout']).to(DEV)
    tl, vl = make_loader(train_ids, True, cfg), make_loader(val_ids, False, cfg)
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])        # SPAN: Adam, no weight decay
    state = dict(epoch=0, best_cer=1e9, best_epoch=0, bad_epochs=0, sessions=0)
    last = f'{run_dir}/last.pt'
    if os.path.exists(last):
        ck = torch.load(last, map_location=DEV, weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); state = ck['state']
        state['sessions'] = state.get('sessions', 0) + 1
        print(f"RESUMED at epoch {state['epoch']+1} (session #{state['sessions']+1}). "
              f"Best val CER {state['best_cer']:.2f} at epoch {state['best_epoch']}.")
    json.dump(cfg, open(f'{run_dir}/config.json','w'), indent=2)
    log = f'{run_dir}/log.csv'
    COLS = ['epoch','train_loss','val_CER','val_WER','val_median_CER','val_catastrophic_pct',
            'val_chrF_pp','blank_slot_pct','pred_len','gt_len','grad_norm','weight_norm',
            'batch_w_mean','skipped','sec','elapsed_h']
    if not os.path.exists(log):
        csv.writer(open(log,'w',newline='')).writerow(COLS)
    gt_len = float(np.mean([len(text[n]) for n in val_ids]))

    while state['epoch'] < cfg['max_epochs'] and state['bad_epochs'] < cfg['patience']:
        if budget_left() < 0.3:
            print(f"TIME BUDGET reached ({elapsed_h():.1f} h) at epoch {state['epoch']}. "
                  f"last.pt is saved - attach this notebook's output as an input and commit "
                  f"again to continue at epoch {state['epoch']+1}."); break
        t0 = time.time(); model.train(); tot = nb = skipped = 0; gmax = 0.0; wsum = 0.0
        for x, y, yl, ws, _ in tl:
            seq, g = model(x.to(DEV, non_blocking=True))
            logp = seq.float().log_softmax(-1).permute(1, 0, 2)
            loss = CTC(logp, y, full_lens(seq), yl)
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True); skipped += 1; continue
            opt.zero_grad(set_to_none=True); loss.backward()
            gmax = max(gmax, float(torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)))
            opt.step()
            tot += loss.item(); nb += 1; wsum += x.shape[-1]
        train_loss = tot/max(1, nb)
        wnorm = float(sum(p.detach().float().pow(2).sum() for p in model.parameters()).sqrt())
        if skipped > 0.05*len(tl):
            print(f'ABORTED: {skipped}/{len(tl)} batches gave a non-finite loss - training has '
                  f'destabilised (weight norm {wnorm:,.0f}).'); break
        refs, hyps, _, blank = evaluate(model, vl); m = compute_metrics(refs, hyps)
        state['epoch'] += 1
        improved = m['CER'] < state['best_cer'] - cfg['min_delta']
        if improved:
            state['best_cer'] = m['CER']; state['best_epoch'] = state['epoch']; state['bad_epochs'] = 0
            torch.save({'model':model.state_dict(),'cfg':cfg,'epoch':state['epoch'],'val':m},
                       f'{run_dir}/best.pt')
        else:
            state['bad_epochs'] += 1
        torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'state':state}, last)
        sec = time.time()-t0
        pred_len = float(np.mean([len(h) for h in hyps])); bw = wsum/max(1, nb)
        csv.writer(open(log,'a',newline='')).writerow(
            [state['epoch'], round(train_loss,4), round(m['CER'],2), round(m['WER'],2),
             round(m['median_CER'],2), round(m['catastrophic_pct'],1), round(m['chrF_pp'],2),
             round(100*blank,1), round(pred_len,1), round(gt_len,1), round(gmax,2),
             round(wnorm,1), round(bw), skipped, round(sec), round(elapsed_h(),3)])
        if state['epoch'] % verbose_every == 0 or improved:
            tag = '  *BEST*' if improved else f"  ({state['bad_epochs']}/{cfg['patience']} stale)"
            print(f"ep {state['epoch']:4d} | loss {train_loss:6.3f} | CER {m['CER']:6.2f} "
                  f"(med {m['median_CER']:5.1f}) | WER {m['WER']:6.2f} | chrF {m['chrF_pp']:5.1f} | "
                  f"best {state['best_cer']:6.2f}@{state['best_epoch']:<4d} | blank {100*blank:4.1f}% | "
                  f"len {pred_len:5.1f}/{gt_len:.0f} | g {gmax:5.1f} w {wnorm:6.0f} | "
                  f"{sec:4.0f}s | {elapsed_h():4.1f}h/{BUDGET_H:.0f}h{tag}")
            print('   GT  :', refs[0][:85]); print('   PRED:', hyps[0][:85])
        if stop_at_cer is not None and m['CER'] <= stop_at_cer:
            print(f'reached CER {m["CER"]:.2f}, moving on'); break
    print(f"done: {state['epoch']} epochs, best val CER {state['best_cer']:.2f} "
          f"at epoch {state['best_epoch']}")
    return model
print('ready')

In [ ]:
# Cell 13 - SAFEGUARD: can it memorise 20 patches? This proves images, labels, alphabet, CTC
# lengths, decoding and metrics are wired up correctly before the long run starts.
#
# lr 1e-3 here, NOT the 1e-4 of the main run. That is deliberate and was learned the hard way:
# written as dict(CFG, ...) without overriding lr, this test silently inherited the main learning
# rate, took 140 epochs just to leave the prior, and failed. Notebook 01 chose 1e-3 for this test
# and reached CER ~1. A memorisation test wants updates, not patience.
smoke = dict(CFG, run_name='smoke', use_aug=False, batch_size=2, lr=1e-3,
             max_epochs=500, patience=500)
ids20 = SPLIT_IDS['train'][:20]
_ = fit(smoke, ids20, ids20, f'{OUT}/runs/smoke', verbose_every=25, stop_at_cer=5.0)

# THE GATE. Without it a headless commit walks straight into the long run on a broken pipeline.
# It checks three things, because CER alone once passed a model that was still emitting blanks:
# a model that is not reading scores about 84 CER, sits at the prior (3.32 nats) and is >90% blank.
SMOKE_MAX_CER, SMOKE_MAX_LOSS, SMOKE_MAX_BLANK = 10.0, 2.50, 90.0
_sl = pd.read_csv(f'{OUT}/runs/smoke/log.csv'); _bi = _sl['val_CER'].idxmin()
s_cer, s_loss, s_blank = (float(_sl.loc[_bi,'val_CER']), float(_sl.loc[_bi,'train_loss']),
                          float(_sl.loc[_bi,'blank_slot_pct']))
print(f'smoke test best epoch: CER {s_cer:.2f} (need <= {SMOKE_MAX_CER}) | '
      f'loss {s_loss:.2f} (need <= {SMOKE_MAX_LOSS}, the prior is 3.32) | '
      f'blank {s_blank:.1f}% (need <= {SMOKE_MAX_BLANK})')
assert s_cer <= SMOKE_MAX_CER and s_loss <= SMOKE_MAX_LOSS and s_blank <= SMOKE_MAX_BLANK, (
    f'SMOKE TEST FAILED: CER {s_cer:.1f}, loss {s_loss:.2f}, blank {s_blank:.1f}% after '
    f'{len(_sl)} epochs on 20 patches it was allowed to memorise. Check runs/smoke/log.csv: if '
    f'CER was still falling steeply at the end it needed more epochs; if it sat near 84 CER with '
    f'blank above 90% the whole way, something IS broken (images, labels, CTC lengths, decoding).')
print('smoke test passed - the pipeline can learn. Starting the long run.')

In [ ]:
# Cell 14 - the run. One stage, from scratch, exactly as SPAN does it.
#
# No curriculum, no two-phase warm start, no easy-patch filter. SPAN's own ablation trains from
# scratch on paragraphs and reports 6.46% CER on IAM (versus 5.45% with line pre-training, which
# we cannot do because MoDeTrans has no line-level labels). The earlier attempts at a curriculum
# were working around a problem that was really the BiLSTM making epochs too expensive to run
# enough of them.
#
# WHAT TO EXPECT, so nobody panics and kills a healthy run again:
#   * the first epochs sit near loss 3.32, the letter-frequency prior, with blank near 100%.
#     This is normal CTC behaviour and is NOT a failure.
#   * SPAN and VAN both train for 5,000 epochs. Judge progress over hundreds of epochs, not tens.
#   * the honest signs of life are: blank falling, pred_len climbing toward gt_len, chrF rising.
#
# It stops on the 11 h budget and resumes next session from last.pt.
print(f'{len(SPLIT_IDS["train"])} training patches, batch {CFG["batch_size"]} -> '
      f'{len(SPLIT_IDS["train"])//CFG["batch_size"]} optimiser updates per epoch')
model = fit(CFG, SPLIT_IDS['train'], SPLIT_IDS['val'][:150], RUN_DIR, verbose_every=1)

_l = pd.read_csv(f'{RUN_DIR}/log.csv')
if len(_l) > 2:
    per_ep = _l['sec'].tail(20).mean()
    print(f'\n{per_ep:.0f} s per epoch -> about {11*3600/per_ep:.0f} epochs per 11 h session, '
          f'so {CFG["max_epochs"]} epochs needs roughly '
          f'{CFG["max_epochs"]/(11*3600/per_ep):.1f} sessions.')

In [ ]:
# Cell 15 - final score on the held-out test split, using the best checkpoint
src = f'{RUN_DIR}/best.pt'
assert os.path.exists(src), 'no checkpoint yet - the run has not completed an improving epoch'
best = torch.load(src, map_location=DEV, weights_only=False)
model = SPAN(NUM_CLASSES, CFG['d_model'], CFG['dropout']).to(DEV)
model.load_state_dict(best['model'])
refs, hyps, ns, blank = evaluate(model, make_loader(SPLIT_IDS['test'], False))
res = compute_metrics(refs, hyps, n_boot=10000)
res.update(run=CFG['run_name'], from_checkpoint=src, best_epoch=best['epoch'],
           val_CER=best['val']['CER'], hours=round(elapsed_h(), 2),
           blank_slot_pct=round(100*blank, 1))
json.dump(res, open(f'{OUT}/test_metrics.json','w',encoding='utf-8'), indent=2, ensure_ascii=False)
pd.DataFrame({'image_id':ns,'reference':refs,'prediction':hyps,
              'cer':[editdistance.eval(_u(r),_u(h))/max(1,len(_u(r))) for r,h in zip(refs,hyps)]}
            ).to_csv(f'{OUT}/test_predictions.csv', index=False, encoding='utf-8')
print(json.dumps(res, indent=2, ensure_ascii=False))

In [ ]:
# Cell 16 - curves and example predictions, saved as files for the report
d = pd.read_csv(f'{RUN_DIR}/log.csv')
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(d.epoch, d.train_loss); ax[0].axhline(3.32, color='r', ls=':', label='letter-frequency prior')
ax[0].set_title('training loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(d.epoch, d.val_CER, label='CER'); ax[1].plot(d.epoch, d.val_WER, label='WER')
ax[1].axhline(84, color='r', ls=':', label='no-reading baseline'); ax[1].legend()
ax[1].set_title('validation error'); ax[1].set_xlabel('epoch')
ax[2].plot(d.epoch, d.blank_slot_pct, label='% blank slots')
ax[2].plot(d.epoch, 100*d.pred_len/d.gt_len, label='predicted length as % of true')
ax[2].legend(); ax[2].set_title('is it actually emitting text?'); ax[2].set_xlabel('epoch')
plt.tight_layout(); plt.savefig(f'{OUT}/curves.png', dpi=120); plt.show()

if os.path.exists(f'{OUT}/test_predictions.csv'):
    p = pd.read_csv(f'{OUT}/test_predictions.csv').sort_values('cer')
    print('\nBEST predictions')
    for _, r in p.head(3).iterrows():
        print(f"  CER {r.cer:.2f}\n   GT  : {str(r.reference)[:90]}\n   PRED: {str(r.prediction)[:90]}")
    print('\nWORST predictions')
    for _, r in p.tail(3).iterrows():
        print(f"  CER {r.cer:.2f}\n   GT  : {str(r.reference)[:90]}\n   PRED: {str(r.prediction)[:90]}")

## Reading the output

**Do not judge this run over tens of epochs.** SPAN and VAN are trained for **5,000**. Every run
we killed for "plateauing" was killed inside its first few percent of training.

| column in `log.csv` | what it tells you |
|---|---|
| `train_loss` | 3.32 is the letter-frequency prior — the score of a model using no image information. Below 3.0 it has started using the picture. |
| `val_CER` | the headline number, micro-averaged over characters. |
| `blank_slot_pct` | share of output slots predicting `<blank>`. Near 100% means it is emitting nothing yet. **Watch this before CER.** |
| `pred_len` / `gt_len` | mean predicted vs true character count. Climbing toward parity is the clearest early sign of life. |
| `val_chrF_pp` | chrF++ rewards partially correct words and rises before CER falls. |
| `grad_norm` / `weight_norm` | stability. Weight norm doubling within a few epochs preceded both earlier blow-ups. |
| `sec` | seconds per epoch. Multiply by 5,000 to see how many sessions the full schedule needs. |

**Baselines to beat.** Random text of the correct length scores ~84 CER; the right letters in the
wrong order also ~84. So anything below ~80 means real signal, and the target for this baseline is
the 40s and below once it has had its epochs.

**Next lever if this is not enough.** The SPAN authors publish pre-trained weights under CC-BY-4.0
(zenodo.org/records/7244546). The FCN encoder is charset-independent — only the final 1×1 head
depends on the alphabet — so those weights can initialise the encoder as genuine transfer learning
from IAM. Their ablation shows pre-training buys about 1 CER point on IAM and 3 on READ 2016. That
is a deliberate, separate experiment; this notebook establishes the from-scratch baseline first.